# File Synthesis: 3 independent datasets → 1 tidy summary CSV each, stacked into one master CSV

## 1. AI_Impact_on_Jobs_2030.csv


In [2]:
import pandas as pd

jobs = pd.read_csv('AI_Impact_on_Jobs_2030.csv')

# synthesis: avg salary / automation risk / AI exposure by Risk_Category x Education_Level
jobs_synth = (jobs.groupby(['Risk_Category', 'Education_Level'])
    .agg(n=('Job_Title', 'size'),
         avg_salary=('Average_Salary', 'mean'),
         avg_automation_prob=('Automation_Probability_2030', 'mean'),
         avg_ai_exposure=('AI_Exposure_Index', 'mean'))
    .reset_index())
jobs_synth.to_csv('jobs_synthesis.csv', index=False)
jobs_synth

,Risk_Category,Education_Level,n,avg_salary,avg_automation_prob,avg_ai_exposure
0,High,Bachelor's,188,87616.675532,0.832074,0.509521
1,High,High School,187,83841.438503,0.825508,0.501711
2,High,Master's,188,90989.228723,0.824681,0.515904
3,High,PhD,177,86947.615819,0.836102,0.512373
4,Low,Bachelor's,190,87420.594737,0.179105,0.498263
5,Low,High School,182,93598.390110,0.181978,0.495659
6,Low,Master's,188,90892.606383,0.168936,0.488457
7,Low,PhD,179,84701.776536,0.176536,0.502067
8,Medium,Bachelor's,387,90082.702842,0.496925,0.489328
9,Medium,High School,415,88154.096386,0.509904,0.496506


## 2. RAG_Context_Adherence_And_Hallucination_Benchmark.csv

In [3]:
rag = pd.read_csv('RAG_Context_Adherence_And_Hallucination_Benchmark.csv')

# synthesis: hallucination rate & faithfulness by model x hallucination type
rag_synth = (rag.groupby(['LLM_Model_Name', 'Hallucination_Type'])
    .agg(n=('Interaction_ID', 'size'),
         hallucination_rate=('Is_Hallucination', 'mean'),
         avg_context_faithfulness=('Context_Faithfulness_Score', 'mean'),
         avg_answer_relevance=('Answer_Relevance_Score', 'mean'))
    .reset_index())
rag_synth.to_csv('rag_synthesis.csv', index=False)
rag_synth

,LLM_Model_Name,Hallucination_Type,n,hallucination_rate,avg_context_faithfulness,avg_answer_relevance
0,Claude-3.5-Sonnet,Contradiction,1281,1.0,0.201830,0.595970
1,Claude-3.5-Sonnet,Fabrication,2013,1.0,0.501982,0.703517
2,Claude-3.5-Sonnet,Unrelated,956,1.0,0.080837,0.179006
3,GPT-4o,Contradiction,1175,1.0,0.197873,0.607830
4,GPT-4o,Fabrication,1958,1.0,0.500986,0.703874
5,GPT-4o,Unrelated,945,1.0,0.077503,0.175533
6,Llama-3-70B,Contradiction,1568,1.0,0.203733,0.594902
7,Llama-3-70B,Fabrication,2211,1.0,0.503434,0.698209
8,Llama-3-70B,Unrelated,1211,1.0,0.081050,0.173502
9,Mistral-7B,Contradiction,1880,1.0,0.197321,0.603461


## 3. salaries.csv

In [5]:
sal = pd.read_csv('salaries.csv')

# synthesis: avg salary by experience_level x company_size x work_year
sal_synth = (sal.groupby(['work_year', 'experience_level', 'company_size'])
    .agg(n=('job_title', 'size'),
         avg_salary_usd=('salary_in_usd', 'mean'),
         avg_remote_ratio=('remote_ratio', 'mean'))
    .reset_index())
sal_synth.to_csv('salaries_synthesis.csv', index=False)
sal_synth

,work_year,experience_level,company_size,n,avg_salary_usd,avg_remote_ratio
0,2020,EN,L,9,85821.555556,72.222222
1,2020,EN,M,2,31362.500000,50.000000
2,2020,EN,S,10,63153.500000,55.000000
3,2020,EX,L,3,234944.333333,83.333333
4,2020,EX,M,1,15000.000000,0.000000
...,...,...,...,...,...,...
60,2025,EX,M,4,179058.750000,0.000000
61,2025,MI,L,12,177209.000000,16.666667
62,2025,MI,M,116,130304.172414,17.241379
63,2025,SE,L,4,197500.000000,0.000000


## 4. Combined download CSV (all 3, tidy long format)

In [6]:
def to_long(df, source, group_cols):
    df = df.copy()
    df['group'] = df[group_cols].astype(str).agg(' | '.join, axis=1)
    df = df.drop(columns=group_cols)
    long = df.melt(id_vars=['group'], var_name='metric', value_name='value')
    long.insert(0, 'source_file', source)
    return long

combined = pd.concat([
    to_long(jobs_synth, 'AI_Impact_on_Jobs_2030.csv', ['Risk_Category', 'Education_Level']),
    to_long(rag_synth, 'RAG_Context_Adherence_And_Hallucination_Benchmark.csv', ['LLM_Model_Name', 'Hallucination_Type']),
    to_long(sal_synth, 'salaries.csv', ['work_year', 'experience_level', 'company_size']),
], ignore_index=True)

combined.to_csv('combined_synthesis.csv', index=False)
combined

,source_file,group,metric,value
0,AI_Impact_on_Jobs_2030.csv,High | Bachelor's,n,188.000000
1,AI_Impact_on_Jobs_2030.csv,High | High School,n,187.000000
2,AI_Impact_on_Jobs_2030.csv,High | Master's,n,188.000000
3,AI_Impact_on_Jobs_2030.csv,High | PhD,n,177.000000
4,AI_Impact_on_Jobs_2030.csv,Low | Bachelor's,n,190.000000
...,...,...,...,...
286,salaries.csv,2025 | EX | M,avg_remote_ratio,0.000000
287,salaries.csv,2025 | MI | L,avg_remote_ratio,16.666667
288,salaries.csv,2025 | MI | M,avg_remote_ratio,17.241379
289,salaries.csv,2025 | SE | L,avg_remote_ratio,0.000000
